In [1]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from pathlib import Path

In [2]:
#Mount Google Drive to access our dataset stored there instead of uploading files repeatedly
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Define the base directory where our Alzheimer's disease neuroimaging dataset is stored
BASE_DIR = '/content/drive/MyDrive/Final_Dataset1'

#Define folder structure based on our dataset organization:
# - Modality types: MRI, PET, co-registered (coreg)
# - Patient classes: Alzheimer's Disease (ad) vs Control Normal (cn)
# - Data augmentation status: original vs augmented

folders = [
    'original_mri_ad', 'original_mri_cn',
    'augmented_mri_ad', 'augmented_mri_cn',
    'original_pet_ad', 'original_pet_cn',
    'augmented_pet_ad', 'augmented_pet_cn',
    'coreg_ad_augmented', 'coreg_ad_original',
    'coreg_cn_augmented', 'coreg_cn_original'
]


In [4]:
#Load neuroimaging data from NIfTI format which is the standard for brain imaging
def load_nifti_file(file_path):
    try:
        img = nib.load(file_path)
        #Convert to numpy array for analysis since we need numerical access to voxel values
        data = img.get_fdata()
        #Some scans might have time as 4th dimension; we only need spatial data for this analysis
        if len(data.shape) == 4:
            data = data[:, :, :, 0]
        return data, img.header
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

In [5]:
#Build a catalogue of dataset entries to streamline loading, validation, and analysis without repeatedly scanning directories.
def datasetCatalogue(base_dir, folders):
    catalogue = []

    for folder in folders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.exists(folder_path):
        #Skip folders that don't exist to allow flexible folder management and avoid runtime errors.
            continue

        is_augmented = 'augmented' in folder #Identify augmented data early to enable separate evaluation if needed.
        is_coregistered = 'coreg' in folder  #Identify co-registered scans because they have different preprocessing assumptions.

        if is_coregistered:
            modality = 'coreg' #Treat co-registered data as a distinct modality for better downstream handling.
            label = 1 if 'ad' in folder else 0 #Infer labels from folder naming convention to avoid manual annotation.
        else:
            modality = 'mri' if 'mri' in folder else 'pet' #Default to standard modalities unless otherwise indicated.
            label = 1 if 'ad' in folder else 0


        nifti_files = [f for f in os.listdir(folder_path) if f.endswith('.nii.gz')]

        for nifti_file in nifti_files:
            file_path = os.path.join(folder_path, nifti_file)

            try:
                img = nib.load(file_path)
                shape = img.shape
                spacing = img.header.get_zooms()
            except Exception as e:
              #Continue processing other files even if some files are corrupt or unreadable, ensuring maximum dataset is used
                print(f"Error looking at {file_path}: {e}")
                continue

            #Consolidate relevant metadata to allow flexible querying, stratification, and data quality analysis later.
            catalogue.append({'file_path': file_path, 'filename': nifti_file, 'folder': folder, 'modality': modality, 'label': label, 'is_augmented': is_augmented, 'is_coregistered': is_coregistered, 'shape': shape,'spacing': spacing})

    df = pd.DataFrame(catalogue)
    return df

catalogue_df = datasetCatalogue(BASE_DIR, folders)

#Provide quick dataset health checks to detect class imbalance, missing modalities, or preprocessing issues early.
print(f"Total files: {len(catalogue_df)}")
print(f"MRI files: {len(catalogue_df[catalogue_df['modality'] == 'mri'])}")
print(f"PET files: {len(catalogue_df[catalogue_df['modality'] == 'pet'])}")
print(f"Co-registered files: {len(catalogue_df[catalogue_df['modality'] == 'coreg'])}")
print(f"AD samples: {len(catalogue_df[catalogue_df['label'] == 1])}")
print(f"CN samples: {len(catalogue_df[catalogue_df['label'] == 0])}")
print(f"Original samples: {len(catalogue_df[catalogue_df['is_augmented'] == False])}")
print(f"Augmented samples: {len(catalogue_df[catalogue_df['is_augmented'] == True])}")


print("\nShape distribution:")
print(catalogue_df.groupby(['modality', 'shape']).size())

Total files: 1026
MRI files: 342
PET files: 342
Co-registered files: 342
AD samples: 432
CN samples: 594
Original samples: 513
Augmented samples: 513

Shape distribution:
modality  shape             
coreg     (128, 128, 128)       342
mri       (128, 128, 128)       342
pet       (128, 128, 128)       340
          (128, 128, 128, 4)      2
dtype: int64
